# 07 · Case cards, symptom lexicon, generated consultations and splits

**SimClinic, CS425 Group 7 (data).** This notebook builds the training data that both models share, following Sections 2–3 of the proposal:

1. `data/processed/cards/cards.jsonl`: about 2,000 case cards (hidden disease, 3–6 symptoms, persona) for 61 conditions, split 80/10/10 **by card** and stratified by disease.
2. `data/processed/dialogues/dialogues.jsonl`: about 5,000 multi-turn consultations generated offline from the cards by a local instruct model (no external API), each checked by the dialogue filter and labelled with a differential of up to three conditions, the true one first.
3. `data/processed/dialogues/review/`: 100 accepted dialogues for the manual check before training.

Notebook 06 holds the audit behind these choices. Shared code is in `simclinic/`: `cards.py` (conditions, cards, splits), `symptom_lexicon.json` + `lexicon.py` (lay phrasings and symptom extraction), and `generate.py` (prompt, filter, generation run).

**Generation runs outside the notebook**, because it takes several hours (it can be stopped and restarted):
```bash
ollama serve &                                   # OLLAMA_NUM_PARALLEL=4 to match --workers
ollama pull qwen2.5:14b-instruct-q4_K_M
python -m simclinic.generate --cards data/processed/cards/cards.jsonl --out-dir data/processed/dialogues
python -m simclinic.generate --cards data/processed/cards/cards.jsonl --out-dir data/processed/dialogues --recheck --max-attempts 15
```
The second command re-applies the current filter to every accepted dialogue and regenerates any that now fail. Run it after changing the lexicon or the filter. Attempts count across runs and some dialogues have used 12, so keep `--max-attempts` above that. Run this notebook after sections 1–3 to write the cards, and again once generation has finished.

## 0. Config

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

CARDS_PER_CONDITION = 33        # 61 conditions x 33 = 2,013 cards (proposal: about 2,000)
SPLIT_RATIOS = (0.8, 0.1, 0.1)  # by card, stratified by disease
REVIEW_N = 100                  # dialogues read by hand before training

RAW_SYMCAT = ROOT / "data" / "raw" / "symcat" / "symcat-801-diseases.csv"
STYLE_JSONL = ROOT / "data" / "processed" / "diagnose-me" / "clean" / "train.jsonl"   # notebook 02
CARDS_JSONL = ROOT / "data" / "processed" / "cards" / "cards.jsonl"
DIALOGUE_DIR = ROOT / "data" / "processed" / "dialogues"
SPLITS = ["train", "dev", "test"]
SEED = 42

## 1. Conditions
59 common Singapore GP conditions that pass the Symcat quality filter, plus the team's dengue and HFMD cards (`simclinic/team_conditions.json`, with sources for every measured percentage). Each condition also has a **course** (acute, subacute or chronic), which decides when the persona's symptoms started, so a cold never started "a month ago". The condition's most common symptom is the **presenting complaint** and is on every card.

In [2]:
import json
from collections import Counter
import pandas as pd
from simclinic import cards as C, symcat

pd.set_option("display.max_colwidth", 100)
conditions = C.load_conditions(RAW_SYMCAT)
failing = [n for n in C.SYMCAT_CONDITIONS if not symcat.passes_quality_filter(conditions[n])]
assert not failing, f"fail the Symcat quality filter: {failing}"
assert set(C.COURSE) == set(C.CONDITIONS), set(C.CONDITIONS) ^ set(C.COURSE)

table = pd.DataFrame([{
    "condition": n, "source": "team card" if n in C.TEAM_CONDITIONS else "Symcat",
    "course": " / ".join(C.COURSE[n]),
    "presenting complaint": "{} {:.0%}".format(*next(iter(conditions[n].symptoms.items()))),
    "symptoms listed": len(conditions[n].symptoms)} for n in C.CONDITIONS])
print(f"{len(C.CONDITIONS)} conditions:", table["source"].value_counts().to_dict(), "| course:", table["course"].value_counts().to_dict())
table

61 conditions: {'Symcat': 59, 'team card': 2} | course: {'acute': 35, 'acute / subacute': 10, 'subacute / chronic': 10, 'chronic': 6}


,condition,source,course,presenting complaint,symptoms listed
0,Flu,Symcat,acute,Fever 91%,12
1,Common cold,Symcat,acute,Cough 85%,12
2,Strep throat,Symcat,acute,Sore throat 95%,12
3,Acute sinusitis,Symcat,acute / subacute,Cough 81%,12
4,Acute otitis media,Symcat,acute,Ear pain 84%,12
...,...,...,...,...,...
56,Aphthous ulcer,Symcat,acute,Mouth ulcer 82%,12
57,Heat exhaustion,Symcat,acute,Dizziness 80%,12
58,Viral exanthem,Symcat,acute,Skin rash 97%,12
59,Dengue fever,team card,acute,Fever 95%,12


## 2. Symptom lexicon
`symptom_lexicon.json` maps every Symcat symptom of the 61 conditions to lay phrasings. The first phrasing is what the generator and the rule-based patient say ("Coryza" → "a runny nose"); all of them are matched in free text. `lexicon.py` also matches the ways patients phrase them inside a sentence:
- "the" and "your" read as "my", and degree words ("really", "a bit") are ignored;
- "sore throat" also matches "my throat feels sore", and "ear pain" matches "my ear hurts";
- "my eyes are red and swollen" counts both states;
- **stop phrases** such as "anxious about" block a shorter match, since "I'm anxious about the results" is the persona's temperament, not the symptom.

**On-card** means the card has the symptom or one that everyday speech does not tell apart from it: "I feel hot" on a card with fever, or "pain in my lower belly" on a card with suprapubic pain (`_same_complaint` and `_covered_by` in the JSON). The dialogue filter and the symptom-fidelity metric both use this test.

The phrasings were extended by reading generated dialogues where a mention was missed or misread.

In [3]:
from simclinic.lexicon import Lexicon, spoken_variants

LEXICON_RAW = json.loads((Path(C.__file__).with_name("symptom_lexicon.json")).read_text(encoding="utf-8"))
symptom_vocab = sorted({s for n in C.CONDITIONS for s in conditions[n].symptoms})
no_entry = [s for s in symptom_vocab if s not in LEXICON_RAW]
assert not no_entry, f"symptoms without a lexicon entry: {no_entry}"
lexicon = Lexicon.load(symptoms=symptom_vocab)

listed = sum(len(lexicon.phrases[s]) for s in symptom_vocab)
variants = sum(len(spoken_variants(p)) for s in symptom_vocab for p in lexicon.phrases[s])
pd.Series({"symptoms (all with an entry)": len(symptom_vocab), "listed phrasings": listed,
           "generated spoken variants": variants,
           "stop phrases": len(LEXICON_RAW["_not_symptoms"]),
           "same-complaint groups": len(LEXICON_RAW["_same_complaint"]),
           "symptoms with a broader on-card rule": len([k for k in LEXICON_RAW["_covered_by"] if not k.startswith("_")])}).to_frame("count")

,count
symptoms (all with an entry),202
listed phrasings,1509
generated spoken variants,12218
stop phrases,8
same-complaint groups,11
symptoms with a broader on-card rule,31


In [4]:
examples = [
    "I've had a runny nose and my throat is really sore since Monday.",
    "No fever, just the cough, and it burns when I pee.",
    "My eyes are red and swollen, and there's stuff coming out of my eye.",
    "I'm a bit anxious about what this could be.",
    "Not really, but I've been feeling low and can't fall asleep.",
    "My hands and feet have been a bit swollen.",
]
pd.DataFrame([(t, ", ".join(f"{s} ({'has' if p else 'denies' if p is False else 'unsure'})" for s, p in lexicon.mentions(t)) or "-")
              for t in examples], columns=["patient says", "lexicon reads"])

,patient says,lexicon reads
0,I've had a runny nose and my throat is really sore since Monday.,"Coryza (has), Sore throat (has)"
1,"No fever, just the cough, and it burns when I pee.","Fever (denies), Cough (has), Painful urination (has)"
2,"My eyes are red and swollen, and there's stuff coming out of my eye.","Eye redness (has), Swollen eye (has), White discharge from eye (has)"
3,I'm a bit anxious about what this could be.,-
4,"Not really, but I've been feeling low and can't fall asleep.","Depression (has), Insomnia (has)"
5,My hands and feet have been a bit swollen.,Peripheral edema (has)


## 3. Case cards
Each card has the presenting complaint plus further symptoms, each drawn independently with its Symcat percentage, resampled until there are 3–6 in total. Symcat counts symptoms per condition rather than per patient, so symptoms that do not fit the persona are left out: pregnancy, menstrual or vaginal symptoms for men, testicular or prostate symptoms for women, pregnancy and menstrual problems after 50, and infant-only signs ("pulling at ears", "irritable infant"). The manual review caught this (an 85-year-old man with pregnancy problems); the 32 affected cards are resampled from their own random stream, so every other card is unchanged. The persona has:
- age and sex weighted by Symcat's odds ratios, adults only;
- ethnicity by Singapore's resident shares;
- an occupation that fits the age;
- temperament, health literacy, onset (from the condition's course) and severity.

The split is 80/10/10 per disease, so every disease appears in every split, and all dialogues of a card stay in that card's split. Card generation is deterministic (seed 42). If dialogues already exist, the cell checks that they were generated from these same cards.

In [5]:
cards = C.split_cards(C.make_cards(conditions, C.CONDITIONS, CARDS_PER_CONDITION, seed=SEED), ratios=SPLIT_RATIOS, seed=SEED)
lines = "".join(json.dumps(c, ensure_ascii=False) + "\n" for c in cards)
if CARDS_JSONL.exists():
    # a changed card must not keep dialogues (or used-up attempts) generated from its old version
    before = {c["card_id"]: c for c in map(json.loads, CARDS_JSONL.read_text(encoding="utf-8").splitlines())}
    changed = {c["card_id"] for c in cards if before.get(c["card_id"]) != c}
    used = {json.loads(line)["card_id"] for name in ("generated.jsonl", "rejected_attempts.jsonl")
            if (DIALOGUE_DIR / name).exists() for line in open(DIALOGUE_DIR / name, encoding="utf-8")}
    if changed & used:
        raise RuntimeError(f"{len(changed & used)} changed cards already have dialogues; remove their records from "
                           f"{DIALOGUE_DIR.relative_to(ROOT)}/*.jsonl (or move the folder away) first")
CARDS_JSONL.parent.mkdir(parents=True, exist_ok=True)
CARDS_JSONL.write_text(lines, encoding="utf-8")
card_by_id = {c["card_id"]: c for c in cards}

assert all(set(c["symptoms"]) <= set(conditions[c["disease"]].symptoms) for c in cards)
assert all(c["symptoms"][0] == next(iter(conditions[c["disease"]].symptoms)) for c in cards)
per_split = pd.DataFrame(cards).groupby("split").agg(cards=("card_id", "size"), diseases=("disease", "nunique")).loc[SPLITS]
assert (per_split["diseases"] == len(C.CONDITIONS)).all(), "every disease must appear in every split"
print(f"{len(cards):,} cards -> {CARDS_JSONL.relative_to(ROOT)}")
per_split

2,013 cards -> data/processed/cards/cards.jsonl


,cards,diseases
split,,
train,1611,61
dev,201,61
test,201,61


In [6]:
persona = pd.DataFrame([c["persona"] for c in cards])
shares = {k: persona[k].value_counts(normalize=True).round(2).to_dict()
          for k in ["sex", "ethnicity", "temperament", "health_literacy", "severity"]}
shares["symptoms per card"] = Counter(len(c["symptoms"]) for c in cards)
shares["age (min / median / max)"] = f"{persona['age'].min()} / {persona['age'].median():.0f} / {persona['age'].max()}"
pd.Series(shares).to_frame("distribution")

,distribution
sex,"{'F': 0.5, 'M': 0.5}"
ethnicity,"{'Chinese': 0.73, 'Malay': 0.15, 'Indian': 0.08, 'Others': 0.04}"
temperament,"{'calm': 0.26, 'anxious': 0.25, 'terse': 0.25, 'chatty': 0.24}"
health_literacy,"{'high': 0.34, 'low': 0.33, 'medium': 0.33}"
severity,"{'mild': 0.35, 'moderate': 0.33, 'severe': 0.32}"
symptoms per card,"{6: 399, 4: 586, 5: 511, 3: 517}"
age (min / median / max),18 / 45 / 88


In [7]:
cards[0]

{'card_id': 'flu-000',
 'disease': 'Flu',
 'symptoms': ['Fever',
  'Cough',
  'Sore throat',
  'Nasal congestion',
  'Headache',
  'Coryza'],
 'persona': {'age': 20,
  'sex': 'M',
  'ethnicity': 'Malay',
  'occupation': 'software engineer',
  'temperament': 'anxious',
  'health_literacy': 'medium',
  'onset': 'this morning',
  'severity': 'moderate'},
 'split': 'train'}

## 4. Generating the consultations
No source has multi-turn consultations tied to a known disease, so `python -m simclinic.generate` writes them with **Qwen2.5-14B-Instruct** (4-bit, run locally by Ollama). We picked a model family different from the LLaMA-3 and Mistral baselines the doctor is compared against. Each dialogue gets its own prompt, which holds:
- the card: the disease, symptoms in lay words with the presenting complaint first, and the persona;
- 3–4 symptoms the patient does **not** have, taken from the conditions naive Bayes confuses most with this card, so the doctor also asks questions answered "no";
- two or three real MedDialog patient messages as **style examples**: informal and hedging, sharing a symptom with the card and naming none of its disease names;
- the history-taking checklist.

The reply is constrained to JSON. Light repairs fix the structure: merging same-role turns, adding a missing opener or closing remark, replacing an unanswered last question, and removing invented names. Then the **filter** rejects any dialogue where:
- the patient or the doctor names the disease;
- the patient claims a symptom that is not on the card;
- the patient reveals fewer than two card symptoms;
- the doctor asks fewer than 4 or more than 12 questions;
- the doctor knows what it has not been told: it asks where a pain is or what it is like before the patient has mentioned any pain, talks about "your pregnancy" before the patient has said she is pregnant, or says "you mentioned" about a symptom the patient never mentioned;
- the doctor asks where a symptom with no place on the body is ("where do you feel the fever / the cough / the dizziness?");
- the doctor asks about a symptom that cannot apply to the patient (testicle pain in a woman, pregnancy in a man);
- an anxious patient never sounds worried without the doctor raising it first ("Are you worried?" "Yes" does not count).

A rejected dialogue is regenerated with a new seed. The main run allowed 3 attempts; two fill passes then gave the dialogues that gave up a 4th, then a 5th and 6th attempt. The doctor checks and the anxious check came from the manual review (section 7, and the paragraph below). Training cards get three dialogues each, and dev and test cards get one.

**Fixes after the manual review.** The review (section 7) found the doctor using the hidden card, and anxious patients who did not sound anxious. The prompt now tells the doctor:
- it knows only what the patient has said;
- to ask whether there is any pain before asking where it is;
- to use the checklist's *site* step only for a pain, rash or swelling.

The temperament also moved to its own line in the patient's style; inside the persona line, only about 17% of anxious patients had sounded worried. On a 24-dialogue pilot these changes raised acceptance from 8 to 20. The symptoms a patient does not have now also leave out any that cannot apply to the persona.

The new checks were applied to every accepted dialogue in two rounds:
- **Round 1.** 1,555 of 5,184 failed: 1,138 anxious patients never sounded worried, 282 doctors presumed a pain, 235 asked "where", and 5 said "you mentioned". Together with the 51 that had given up, 1,606 were regenerated with up to 9 attempts, and 1,579 were accepted.
- **Round 2,** after re-reading the regenerated review dialogues. The generator had passed the worry check by having the doctor ask "Are you worried about what this could be?": 27% of anxious dialogues had that question, against 1% for the other temperaments. The worry must now come unprompted, and 275 dialogues failed that. Another 8 doctors spoke of "your pregnancy" before the patient mentioned one, and 12 asked about symptoms that could not apply. The lexicon also now reads "my neck has been hurting" and "sharp pains in my chest", which failed 3 more. Together with the 27 that had given up, 324 were regenerated with up to 12 attempts, and 318 were accepted.

**Prompt fix between the fill passes.** The main run's prompt showed "my neck hurts too" as an example of naming a symptom, and the model copied it into about 500 rejected attempts, mostly for sore-throat conditions ("Neck pain" in the rejection table below). It never reached an accepted dialogue, because the filter rejects it whenever neck pain is off the card, but it left tonsillitis without a dev dialogue. The example is now built from one of the card's own symptoms ("I've got a fever too").

In [8]:
import random
from simclinic import generate as G
from simclinic.naive_bayes import SymcatNaiveBayes

nb = SymcatNaiveBayes({n: conditions[n].symptoms for n in C.CONDITIONS})
names = {n: C.disease_names(conditions[n]) for n in C.CONDITIONS}
all_names = sorted({x for ns in names.values() for x in ns}, key=len, reverse=True)

# One prompt exactly as the generator builds it (style examples shortened for display).
demo, rng = cards[0], random.Random(SEED)
style = [json.loads(line)["turns"][0]["text"] for _, line in zip(range(20000), open(STYLE_JSONL, encoding="utf-8"))]
index = G.build_style_index(style, lexicon)
examples = [e[:200] for e in G.style_examples(demo, style, index, names[demo["disease"]], rng, k=2)]
print(G.build_prompt(demo, lexicon, examples, G.negative_symptoms(demo, conditions, nb, lexicon, rng)))

Write a realistic consultation between a general practitioner (GP) in Singapore and a patient.

THE PATIENT (the doctor does not know any of this)
- Condition: Flu. The patient does not know this and never names it or any other diagnosis.
- Main complaint: a fever
- Other symptoms the patient has: a cough; a sore throat; a blocked nose; a headache; a runny nose
- Symptoms the patient does NOT have: shortness of breath; wheezing; body aches; sharp chest pain
- Persona: 20-year-old man (Malay), software engineer. Temperament: anxious. Health literacy: medium (uses everyday words).
- The symptoms started this morning. Severity: moderate.

THE DOCTOR
- Does not know the condition or the patient's symptoms, only what the patient has said so far. Opens by asking what brings the patient in.
- Asks ONE short question per turn, 5 to 9 questions in total, following this history-taking checklist for the complaints the patient describes: main complaint, onset and duration, site (of a pain, rash or

In [9]:
accepted = G.read_jsonl(DIALOGUE_DIR / "generated.jsonl")
attempts = G.read_jsonl(DIALOGUE_DIR / "rejected_attempts.jsonl")
planned = G.plan(cards)
tried = {r["dialogue_id"] for r in accepted + attempts}
gave_up = tried - {r["dialogue_id"] for r in accepted}
finished = len(tried) == len(planned)         # every planned dialogue accepted or out of tries
print(f"{len(accepted):,} of {len(planned):,} planned dialogues accepted, {len(gave_up)} gave up"
      + ("" if finished else f", {len(planned) - len(tried):,} not tried yet (generation not finished)"))
pd.Series({
    "generator": Counter(r["model"] for r in accepted).most_common(1)[0][0] if accepted else "-",
    "calls": len(accepted) + len(attempts),
    "dialogues accepted at the first attempt": f"{sum(r['attempt'] == 0 for r in accepted) / max(1, len(tried)):.1%}",
    "acceptance per call": f"{len(accepted) / max(1, len(accepted) + len(attempts)):.1%}",
    "dialogues that gave up": len(gave_up),
    "repairs applied (accepted)": dict(Counter(f for r in accepted for f in r["fixes"])),
}).to_frame("value")

5,229 of 5,235 planned dialogues accepted, 6 gave up

,value
generator,qwen2.5:14b-instruct-q4_K_M
calls,11085
dialogues accepted at the first attempt,48.5%
acceptance per call,47.2%
dialogues that gave up,6
repairs applied (accepted),"{'replaced unanswered last question': 186, 'added closing': 43, 'merged same-role turns': 39}"


In [10]:
import re

def reason(p):
    if p.startswith("off-card"):
        return "patient claims an off-card symptom"
    p = re.sub(r"^(doctor asks where|doctor says the patient mentioned|doctor asks about symptoms that cannot apply):? \[.*\]$", r"\1 ...", p)
    return re.sub(r"^\d+ questions$", "too few or too many questions", re.sub(r"only \d", "only N", p))

pd.concat({
    "rejection reasons (attempts)": pd.Series(Counter(reason(p) for r in attempts for p in r["problems"])),
    "most common off-card symptoms": pd.Series(Counter(s for r in attempts for p in r["problems"] if p.startswith("off-card")
                                                       for s in eval(p.split(": ", 1)[1]))).nlargest(10),
}).to_frame("count")

count
rejection reasons (attempts)  patient claims an off-card symptom                 2592
                              only N card symptoms revealed                      1594
                              too few or too many questions                         1
                              patient names the disease                             2
                              doctor asks where ...                               250
                              anxious patient never sounds worried               1255
                              doctor presumes a pain nobody mentioned             312
                              doctor says the patient mentioned ...                23
                              doctor presumes a pregnancy nobody mentioned         17
                              anxious patient never sounds worried unprompted     397
                              doctor asks about symptoms that cannot apply ...     13
most common off-card symptoms Neck pain                                           497
                              Difficulty breathing                                365
                              Itching of skin                                     353
                              Fatigue                                             175
                              Insomnia                                            113
                              Nausea                                              111
                              Sweating                                             79
                              Chest tightness                                      73
                              Anxiety and nervousness                              66
                              Sneezing                                             64

## 5. Final filter and splits
Every accepted dialogue is checked once more with the current filter, in case the lexicon changed during the run. Run with `--recheck` to regenerate any that fail. Each record carries its card, so the patient model can build its hidden header from it and the doctor model can ignore it. The split comes from the card, so no card is shared between train, dev and test.

In [11]:
final, failed = [], []
for r in accepted:
    card = card_by_id[r["card_id"]]
    turns, problems = G.check_dialogue(r["turns"], card, lexicon, names[card["disease"]], all_names)
    (failed if problems else final).append({
        "dialogue_id": r["dialogue_id"], "card_id": r["card_id"], "split": card["split"], "disease": card["disease"],
        "symptoms": card["symptoms"], "persona": card["persona"], "turns": turns,
        "diagnosis": G.diagnosis_label(turns, card, nb, lexicon, conditions), "source": r["model"], "problems": problems})
print(f"{len(final):,} dialogues pass the current filter, {len(failed)} fail (run generate --recheck to replace them)")
print("names in the diagnosis label:", dict(sorted(Counter(len(r["diagnosis"]) for r in final).items())))
for r in final + failed:
    del r["problems"]

assert all(r["split"] == card_by_id[r["card_id"]]["split"] for r in final)
assert all(r["diagnosis"][0] == r["disease"] and len(set(r["diagnosis"])) == len(r["diagnosis"]) <= 3 for r in final)
assert all(t["role"] == ("doctor" if i % 2 == 0 else "patient") for r in final for i, t in enumerate(r["turns"]))
assert all(r["turns"][-1]["role"] == "doctor" for r in final)
splits = pd.DataFrame(final).groupby("split").agg(dialogues=("dialogue_id", "size"), cards=("card_id", "nunique"),
                                                   diseases=("disease", "nunique")).reindex(SPLITS)
if finished:
    assert (splits["diseases"] == len(C.CONDITIONS)).all(), "every disease must appear in every split"
splits

5,229 dialogues pass the current filter, 0 fail (run generate --recheck to replace them)
names in the diagnosis label: {1: 4, 2: 47, 3: 5178}


,dialogues,cards,diseases
split,,,
train,4827,1611,61
dev,201,201,61
test,201,201,61


In [12]:
final.sort(key=lambda r: (SPLITS.index(r["split"]), r["dialogue_id"]))
out = DIALOGUE_DIR / "dialogues.jsonl"
out.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in final), encoding="utf-8")
print(f"saved {len(final):,} dialogues -> {out.relative_to(ROOT)}")

saved 5,229 dialogues -> data/processed/dialogues/dialogues.jsonl


## 6. What the dialogues look like
- **Revealed symptoms** are read back from the text with the lexicon, which is exactly what the naive Bayes baseline and the auto-mode metrics see.
- **NB top-1 / top-3 on revealed symptoms** says how much diagnostic evidence a dialogue holds.
- **Words per patient turn** and **sounds worried** (without the doctor raising it first) by temperament check that the persona shows: terse patients should be short, chatty ones long, and anxious ones worried. Other temperaments can sound worried too, which is fine.
- **Doctor asks about worry** should not depend on the temperament, since the doctor cannot see it. Asking about the patient's concerns is good practice, but asking only anxious patients would mean the doctor is reading the hidden card.

In [13]:
from simclinic.lexicon import revealed_symptoms

rows = []
for r in final:
    evidence = revealed_symptoms(r["turns"], lexicon)
    ranked = [d for d, _ in nb.rank(evidence, k=3)]
    rows.append({"split": r["split"], "questions": sum(t["role"] == "doctor" for t in r["turns"]) - 1,
                 "card symptoms revealed": sum(bool(evidence.get(s)) for s in r["symptoms"]) / len(r["symptoms"]),
                 "denied symptoms": sum(v is False for v in evidence.values()),
                 "NB top-1": ranked[0] == r["disease"], "NB top-3": r["disease"] in ranked,
                 "words per patient turn": pd.Series([len(t["text"].split()) for t in r["turns"] if t["role"] == "patient"]).mean(),
                 "sounds worried": any(t["role"] == "patient" and G.WORRY.search(t["text"]) and not G.WORRY.search(r["turns"][i - 1]["text"])
                                       for i, t in enumerate(r["turns"])),
                 "doctor asks about worry": any(t["role"] == "doctor" and G.WORRY.search(t["text"]) and "?" in t["text"] for t in r["turns"]),
                 "temperament": r["persona"]["temperament"]})
stats = pd.DataFrame(rows)
stats.drop(columns=["temperament", "sounds worried", "doctor asks about worry"]).groupby("split").mean().reindex(SPLITS).round(2)

,questions,card symptoms revealed,denied symptoms,NB top-1,NB top-3,words per patient turn
split,,,,,,
train,6.66,0.74,1.55,0.70,0.92,11.11
dev,6.62,0.75,1.46,0.68,0.92,11.02
test,6.55,0.73,1.53,0.64,0.91,11.06


In [14]:
stats.groupby("temperament").agg(dialogues=("split", "size"), words_per_patient_turn=("words per patient turn", "mean"),
                                 median_words=("words per patient turn", "median"),
                                 sounds_worried=("sounds worried", "mean"),
                                 doctor_asks_about_worry=("doctor asks about worry", "mean")).round(2)

,dialogues,words_per_patient_turn,median_words,sounds_worried,doctor_asks_about_worry
temperament,,,,,
anxious,1329,14.12,14.00,1.00,0.08
calm,1367,10.60,10.40,0.04,0.02
chatty,1251,16.15,15.83,0.06,0.02
terse,1282,3.61,3.43,0.02,0.02


In [15]:
def show(r):
    p = r["persona"]
    print(f"[{r['dialogue_id']}] {r['disease']} | card: {'; '.join(lexicon.lay(s) for s in r['symptoms'])}")
    print(f"  persona: {p['age']} {p['sex']} {p['ethnicity']} {p['occupation']}, {p['temperament']}, literacy {p['health_literacy']}, since {p['onset']}")
    for t in r["turns"]:
        print(f"  {t['role']:>7}: {t['text']}")
    print(f"  diagnosis label: {'; '.join(r['diagnosis'])}\n")

for r in random.Random(SEED).sample(final, 3):
    show(r)

[chickenpox-020-d1] Chickenpox | card: a rash; a fever; a cough
  persona: 23 M Chinese sales executive, anxious, literacy medium, since last night
   doctor: What brings you in today?
  patient: I've got a rash and a bit of a fever.
   doctor: When did the rash start?
  patient: It started last night, just after I went to bed.
   doctor: Where on your body is the rash?
  patient: It's all over my body, but mostly on my arms and legs.
   doctor: Does it feel itchy or painful?
  patient: It doesn't feel itchy, just uncomfortable. I didn't feel it was painful until now.
   doctor: Have you had any coughing or other symptoms?
  patient: Yeah, I've got a cough too, just a little one.
   doctor: Do you have any other symptoms like a blocked nose or sore throat?
  patient: No, I don't have a blocked nose or sore throat. I'm just worried about what it could be.
   doctor: Thank you for letting me know about your symptoms. We'll need to keep an eye on how you're doing.
  diagnosis label: Chick

## 7. Manual review sample
Before training, the team reads 100 accepted dialogues: at least one per disease, the rest at random. For each, check that the patient stays in character and on the card, the doctor asks one sensible question at a time, and nothing gives the diagnosis away. `review_sample.md` is for reading; `review_sheet.csv` is for marks and notes (1–5 for staying in character, the same scale as the proposal's rubric). Re-running this cell keeps the sample and the marks. A dialogue regenerated since it was marked loses its marks (the `version` column is a checksum of its turns), and one that no longer exists is replaced by another of the same disease.

In [16]:
import csv
import zlib

def version(r):
    """Checksum of the turns, so marks are dropped when a dialogue is regenerated under the same id."""
    return f"{zlib.crc32(json.dumps(r['turns'], ensure_ascii=False).encode()):08x}"

review_dir = DIALOGUE_DIR / "review"
review_dir.mkdir(exist_ok=True)
sheet_path = review_dir / "review_sheet.csv"
COLUMNS = ["n", "dialogue_id", "disease", "version", "in_character_1to5", "on_card_yn", "doctor_ok_yn",
           "no_giveaway_yn", "reviewer", "notes"]
sheet = list(csv.DictReader(open(sheet_path, newline="", encoding="utf-8"))) if sheet_path.exists() else []

rng = random.Random(SEED)
by_id = {r["dialogue_id"]: r for r in final}
by_disease = {}
for r in final:
    by_disease.setdefault(r["disease"], []).append(r)
if sheet:
    # keep the dialogues already under review; replace one that no longer exists with another of its disease
    review, taken = [], {row["dialogue_id"] for row in sheet}
    for row in sheet:
        r = by_id.get(row["dialogue_id"]) or rng.choice(
            [r for r in by_disease[row["disease"]] if r["dialogue_id"] not in taken])
        taken.add(r["dialogue_id"])
        review.append(r)
else:
    review = [rng.choice(by_disease[d]) for d in sorted(by_disease)]
    rest = [r for r in final if r not in review]
    review += rng.sample(rest, max(0, min(REVIEW_N - len(review), len(rest))))
marks = {row["dialogue_id"]: row for row in sheet if row["dialogue_id"] in by_id
         and row.get("version") == version(by_id[row["dialogue_id"]])}

with open(review_dir / "review_sample.md", "w", encoding="utf-8") as f:
    f.write("# SimClinic: 100 generated dialogues for the manual check\n\n"
            "For each dialogue: is the patient in character and on the card, does the doctor ask one sensible "
            "question at a time, and does anything give the diagnosis away? Record marks in review_sheet.csv.\n")
    for i, r in enumerate(review, 1):
        p = r["persona"]
        f.write(f"\n## {i}. {r['dialogue_id']} ({r['split']})\n\n"
                f"**Hidden disease:** {r['disease']}  \n**Card:** {'; '.join(lexicon.lay(s) for s in r['symptoms'])}  \n"
                f"**Persona:** {p['age']}, {p['sex']}, {p['ethnicity']}, {p['occupation']}, {p['temperament']}, "
                f"health literacy {p['health_literacy']}, since {p['onset']}, {p['severity']}\n\n")
        for t in r["turns"]:
            f.write(f"- **{t['role'].title()}:** {t['text']}\n")
        f.write(f"\n*Diagnosis label:* {'; '.join(r['diagnosis'])}\n")
with open(sheet_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, COLUMNS)
    w.writeheader()
    for i, r in enumerate(review, 1):
        w.writerow({**dict.fromkeys(COLUMNS, ""), **marks.get(r["dialogue_id"], {}),
                    "n": i, "dialogue_id": r["dialogue_id"], "disease": r["disease"], "version": version(r)})
marked = sum(any(marks.get(r["dialogue_id"], {}).get(k) for k in COLUMNS[4:8]) for r in review)
print(f"{len(review)} dialogues ({len({r['disease'] for r in review})} diseases) -> {review_dir.relative_to(ROOT)}, "
      f"{marked} already marked")

100 dialogues (61 diseases) -> data/processed/dialogues/review, 100 already marked


In [17]:
sheet = pd.read_csv(sheet_path)
done = sheet.dropna(subset=["in_character_1to5"])
pd.Series({
    "dialogues marked": f"{len(done)} of {len(sheet)}",
    "in character (mean, 1-5)": round(done["in_character_1to5"].mean(), 2),
    "patient stays on the card": f"{(done['on_card_yn'] == 'Y').mean():.0%}",
    "doctor asks sensibly": f"{(done['doctor_ok_yn'] == 'Y').mean():.0%}",
    "nothing gives the diagnosis away": f"{(done['no_giveaway_yn'] == 'Y').mean():.0%}",
}).to_frame("manual review")

,manual review
dialogues marked,100 of 100
"in character (mean, 1-5)",4.28
patient stays on the card,97%
doctor asks sensibly,93%
nothing gives the diagnosis away,100%


**What the review found** (2026-10-27, re-read after the fixes; per-dialogue notes in `review_sheet.csv`). The first read scored 4.21 for staying in character, 96% on the card, 88% for the doctor and 100% for no giveaway. Dialogues replaced by the fixes below lost their marks. The 28 replacements were read again, giving the totals above.
- **Fixed: cards that did not fit the persona.** One card gave an 85-year-old man pregnancy problems. 32 of the 2,013 cards had a symptom for the wrong sex, for the wrong age, or for infants only. `cards.py` now leaves these out (section 3), and their 80 dialogues were regenerated. The symptoms a patient does *not* have are now checked the same way, after a man was asked whether he was pregnant.
- **Fixed: the doctor using the hidden card in ways a check can see** (section 4):
  - asking where the pain is before any pain was mentioned;
  - saying "you mentioned" about something unsaid;
  - talking about "your pregnancy" to a patient who never mentioned one;
  - asking "where do you feel the fever / the dizziness?" (from the checklist's *site* step).
- **Not fixed: the doctor using the hidden card in ways no check can see** (5 of the 7 "doctor N" marks):
  - it asks about a hidden symptom out of nowhere ("hair loss" for a rash, "retaining more fluid");
  - it names where a hidden pain is ("the pain just above your pubic bone" to a patient who said it hurts to pee).

  Detectors for both flagged mostly sound dialogues (fewer than a third of the hits were real), so these dialogues stay. When one model writes both sides from the card, some of this cannot be avoided, and the doctor model will learn a little of it.
- **Fixed: anxious patients who did not sound anxious** (most of the 3s in the first read). Every anxious patient now says, without being asked, that they are worried. The doctor asks about worry in 8% of anxious dialogues against 2% of the others, and in 100 of those 108 dialogues the patient had already raised it (section 6).
- **The patient side holds up.** Terse and chatty personas clearly differ. Four dialogues add a minor off-card complaint the lexicon misses (a swollen ear, ears "stuffed up"). One dialogue from before the negatives fix has the patient deny joint pain despite knee and ankle pain on the card.
- **Nothing gives the diagnosis away.**
- **Fixed: noisy diagnosis labels.** The 2nd and 3rd places come from naive Bayes on the revealed symptoms. They now have to explain at least a third of those symptoms, which drops pairs like depression → tonsillitis and conjunctivitis → lactose intolerance. As a result, 51 labels have fewer than three names.
- **Fixed: the BPPV name.** It is now "Benign paroxysmal positional vertigo". Symcat spells it *vertical*, and its slug, and so the card and dialogue ids, keep that spelling.
- **Symcat quirks kept as they are:**
  - some cards combine unrelated symptoms (a cold sore with swollen hands, insomnia with seizures); in one dialogue the patient reports a seizure and the doctor moves on, marked unsafe;
  - some personas are odd (a nurse with low health literacy, heat exhaustion lasting five days).

## 8. Next steps
- **Doctor model (notebook 03):** `CARDS_JSONL` and `DIALOGUES_JSONL` point at the files written here. Re-run 03 → 05.
- **Patient model:** each record in `dialogues.jsonl` carries its card (`disease`, `symptoms`, `persona`), so the hidden header from the proposal can be built directly, e.g. `<|disease|>Flu<|symptoms|>a fever; a cough<|persona|>34, F, Chinese, teacher, anxious`, with the loss on patient turns only (notebook 01, steps 7–10).
- **Evaluation:** auto mode uses the 201 test cards. Symptom fidelity should use `Lexicon.on_card`, the same test as the dialogue filter.